In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from experiments.paths import PVALUE_FIGURES_DIR, ensure_dir

# -----------------------------
# 1. Simulation setup
# -----------------------------
n_values = [50, 100, 500, 1000, 5000, 10000, 50000]
mean_shifts = [0, 0.01, 0.05, 0.1, 0.25]
iterations = 200

distributions = {
    "normal": lambda n, delta: (
        np.random.normal(0, 1, n),
        np.random.normal(delta, 1, n),
    ),
    "uniform": lambda n, delta: (
        np.random.uniform(0, 1, n),
        np.random.uniform(delta, 1 + delta, n),  # same width, shifted support
    ),
    "exponential": lambda n, delta: (
        np.random.exponential(1, n),
        np.random.exponential(1 / (1 + delta), n),
    ),
}

# -----------------------------
# 2. Run simulation (median + IQR)
# -----------------------------
records = []
eps = 1e-300  # log-scale safety

for dist_name, sampler in distributions.items():
    for delta in mean_shifts:
        for n in n_values:
            p_t, p_mw, p_ks = [], [], []

            for _ in range(iterations):
                x, y = sampler(n, delta)

                if dist_name == "normal":
                    p_t.append(stats.ttest_ind(x, y, equal_var=True).pvalue)

                p_mw.append(stats.mannwhitneyu(x, y, alternative="two-sided").pvalue)
                p_ks.append(stats.ks_2samp(x, y).pvalue)

            p_mw = np.clip(np.asarray(p_mw, float), eps, 1.0)
            p_ks = np.clip(np.asarray(p_ks, float), eps, 1.0)
            p_t = np.clip(np.asarray(p_t, float), eps, 1.0) if dist_name == "normal" else np.asarray([np.nan])

            def qstats(arr):
                arr = arr[~np.isnan(arr)]
                if arr.size == 0:
                    return np.nan, np.nan, np.nan
                q25, q50, q75 = np.quantile(arr, [0.25, 0.5, 0.75])
                return q50, q25, q75

            t_med, t_q25, t_q75 = qstats(p_t) if dist_name == "normal" else (np.nan, np.nan, np.nan)
            mw_med, mw_q25, mw_q75 = qstats(p_mw)
            ks_med, ks_q25, ks_q75 = qstats(p_ks)

            records.append({
                "dist": dist_name, "delta": delta, "n": n,
                "p_t_med": t_med,   "p_t_q25": t_q25,   "p_t_q75": t_q75,
                "p_mw_med": mw_med, "p_mw_q25": mw_q25, "p_mw_q75": mw_q75,
                "p_ks_med": ks_med, "p_ks_q25": ks_q25, "p_ks_q75": ks_q75,
            })

df = pd.DataFrame(records)
print("✅ Simulation complete:", df.shape, "rows")

# -----------------------------
# Global plot style
# -----------------------------
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "legend.title_fontsize": 12,
})

tests = {
    "Kolmogorov–Smirnov": ("p_ks_med", "p_ks_q25", "p_ks_q75"),
    "Mann–Whitney U":     ("p_mw_med", "p_mw_q25", "p_mw_q75"),
    "t-test":             ("p_t_med",  "p_t_q25",  "p_t_q75"),
}

colors = plt.cm.viridis(np.linspace(0, 1, len(mean_shifts)))

active_distribs_all = ["normal", "uniform", "exponential"]
ensure_dir(PVALUE_FIGURES_DIR)

for test_name, (med_col, q25_col, q75_col) in tests.items():
    active_distribs = ["normal"] if test_name == "t-test" else active_distribs_all
    ncols = len(active_distribs)

    # ---- slightly wider single-panel so legend-outside still looks good
    fig_w = 7.6 if ncols == 1 else 6.6 * ncols
    fig_h = 4.8

    fig, axes = plt.subplots(1, ncols, figsize=(fig_w, fig_h), sharex=True, sharey=True)
    axes = np.atleast_1d(axes).ravel()

    # ---- global y-floor per test (based on q25)
    finite_min = np.inf
    for dname in active_distribs:
        vals = df[df["dist"] == dname][q25_col].to_numpy()
        vals = vals[np.isfinite(vals)]
        if vals.size:
            finite_min = min(finite_min, vals.min())
    ymin = max(1e-16, 10 ** np.floor(np.log10(finite_min if np.isfinite(finite_min) else 1e-16)))

    legend_handles, legend_labels = None, None

    for i, dist_name in enumerate(active_distribs):
        ax = axes[i]
        sub = df[df["dist"] == dist_name]

        for color, delta in zip(colors, mean_shifts):
            subset = sub[sub["delta"] == delta].sort_values("n")
            if subset.empty:
                continue

            n_vals  = subset["n"].to_numpy()
            med_raw = np.clip(subset[med_col].to_numpy(), eps, 1.0)
            q25_raw = np.clip(subset[q25_col].to_numpy(), eps, 1.0)
            q75_raw = np.clip(subset[q75_col].to_numpy(), eps, 1.0)

            med_plot = np.maximum(med_raw, ymin)
            q25_plot = np.maximum(q25_raw, ymin)
            q75_plot = np.maximum(q75_raw, ymin)

            ax.plot(n_vals, med_plot, marker="o", markersize=6, linewidth=2.0,
                    color=color, label=f"Δ={delta}")
            ax.fill_between(n_vals, q25_plot, q75_plot, alpha=0.20, color=color)

            censored = med_raw < ymin
            if np.any(censored):
                ax.scatter(n_vals[censored], np.full(np.sum(censored), ymin),
                           marker="v", s=90, color=color, edgecolor="black",
                           linewidth=0.5, zorder=6)

        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_ylim(ymin, 1.0)
        ax.set_title(dist_name.capitalize())
        ax.grid(True, which="both", ls="--", lw=0.4, alpha=0.7)
        ax.axhline(0.05, color="gray", ls="--", lw=1.0, alpha=0.6)

        if legend_handles is None:
            legend_handles, legend_labels = ax.get_legend_handles_labels()

    # -----------------------------
    # Layout FIX (no tight_layout)
    # -----------------------------
    # Reserve: bottom for xlabel+footnote, right for legend.
    # Works cleanly for 1 or many panels.
    fig.subplots_adjust(
        left=0.10,
        right=0.80,   # space for legend outside
        bottom=0.20,  # space for xlabel + footnote
        top=0.86,
        wspace=0.25
    )

    fig.suptitle(f"P-value Sensitivity to Sample Size ({test_name})", fontsize=15)

    # Use fig.text instead of supxlabel/supylabel to avoid layout collisions
    fig.text(0.5, 0.10, "Sample size (log scale)", ha="center", va="center")
    fig.text(0.02, 0.52, f"{test_name} median p-value (log scale)",
             rotation=90, ha="center", va="center")

    fig.text(0.10, 0.06,
             "Median ± IQR shown; ▼ indicates median p-value below plot floor",
             fontsize=10, alpha=0.8, ha="left")

    if legend_handles:
        fig.legend(
            legend_handles, legend_labels,
            title="Parameter shift Δ",
            loc="center left",
            bbox_to_anchor=(0.82, 0.52),  # outside, in reserved right margin
            frameon=True
        )

    fname = PVALUE_FIGURES_DIR / f"pvalue_sensitivity_{test_name.replace(' ', '_').replace('–', '-')}_subset"
    plt.savefig(str(fname) + ".png", dpi=300)
    plt.savefig(str(fname) + ".pdf")
    print(f"📊 Saved: {fname}.png / .pdf")
    plt.show()

print("✅ Subset plots (Normal / Uniform / Exponential) complete.")


In [ ]:
from experiments.paths import EXPERIMENTS_DIR

df.to_csv(PVALUE_FIGURES_DIR / "p_value_sensitiv.csv", index=False)


In [5]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import jensenshannon
import matplotlib.pyplot as plt

n_values = [50, 100, 500, 1000, 5000, 10000, 50000]
iterations = 200
mean_shift = 0.05
eps = 1e-12

# --- helper functions ---
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    sx, sy = np.var(x, ddof=1), np.var(y, ddof=1)
    sp = np.sqrt(((nx-1)*sx + (ny-1)*sy) / (nx + ny - 2))
    return (np.mean(x) - np.mean(y)) / sp

def js_divergence(x, y, bins=100):
    hist_x, _ = np.histogram(x, bins=bins, range=(min(x.min(), y.min()), max(x.max(), y.max())), density=True)
    hist_y, _ = np.histogram(y, bins=bins, range=(min(x.min(), y.min()), max(x.max(), y.max())), density=True)
    hist_x = np.clip(hist_x, eps, None)
    hist_y = np.clip(hist_y, eps, None)
    return jensenshannon(hist_x, hist_y, base=2) ** 2  # JS divergence (not distance)

distributions = {
    'normal': lambda n: (
        np.random.normal(0, 1, n),
        np.random.normal(mean_shift, 1, n)
    ),
    'uniform': lambda n: (
        np.random.uniform(0, 1, n),
        np.random.uniform(0, 1 + mean_shift, n)
    ),
    'exponential': lambda n: (
        np.random.exponential(1, n),
        np.random.exponential(1 / (1 + mean_shift), n)
    ),
}

# --- simulation ---
records = []
for dist_name, sampler in distributions.items():
    for n in n_values:
        d_vals, js_vals = [], []
        for _ in range(iterations):
            x, y = sampler(n)
            d_vals.append(cohens_d(x, y))
            js_vals.append(js_divergence(x, y))
        records.append({
            'dist': dist_name,
            'n': n,
            'd_mean': np.mean(d_vals),
            'd_std': np.std(d_vals),
            'js_mean': np.mean(js_vals),
            'js_std': np.std(js_vals)
        })

df_ = pd.DataFrame(records)
print(df_.head())


     dist     n    d_mean     d_std   js_mean    js_std
0  normal    50 -0.059983  0.215588  0.547697  0.078336
1  normal   100 -0.054673  0.135713  0.328850  0.049612
2  normal   500 -0.042135  0.059105  0.069879  0.009387
3  normal  1000 -0.051000  0.047999  0.034609  0.005213
4  normal  5000 -0.050620  0.017752  0.007530  0.001140


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from experiments.paths import EFFECT_SIZE_FIGURES_DIR, ensure_dir

# -----------------------------
# 1. Experiment setup
# -----------------------------
n_values = [50, 100, 500, 1000, 5000, 10000, 50000]
mean_shifts = [0.0, 0.01, 0.05, 0.1, 0.25]
iterations = 200
np.random.seed(42)

# -----------------------------
# 2. Helpers
# -----------------------------
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    sx, sy = np.var(x, ddof=1), np.var(y, ddof=1)
    sp2 = ((nx - 1) * sx + (ny - 1) * sy) / (nx + ny - 2)
    sp = np.sqrt(sp2) if sp2 > 0 else np.nan
    return (np.mean(x) - np.mean(y)) / sp

def qstats(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.nan, np.nan, np.nan
    q25, q50, q75 = np.quantile(arr, [0.25, 0.5, 0.75])
    return q50, q25, q75

# -----------------------------
# 3. Distributions (as you used)
# -----------------------------
distributions = {
    'normal': lambda n, delta: (
        np.random.normal(0, 1, n),
        np.random.normal(delta, 1, n)
    ),
    'uniform': lambda n, delta: (
        np.random.uniform(0, 1, n),
        np.random.uniform(0, 1 + delta, n)
    ),
    'exponential': lambda n, delta: (
        np.random.exponential(1, n),
        np.random.exponential(1 / (1 + delta), n)
    ),
}

# -----------------------------
# 4. Simulation (store robust summaries)
# -----------------------------
records = []
for dist_name, sampler in distributions.items():
    for delta in mean_shifts:
        for n in n_values:
            d_vals = []
            for _ in range(iterations):
                x, y = sampler(n, delta)
                d_vals.append(cohens_d(x, y))

            d_med, d_q25, d_q75 = qstats(d_vals)
            d_mean = np.nanmean(d_vals)
            d_std  = np.nanstd(d_vals)

            records.append({
                'dist': dist_name,
                'delta': delta,
                'n': n,
                'd_med': d_med,
                'd_q25': d_q25,
                'd_q75': d_q75,
                'd_mean': d_mean,   # kept (optional)
                'd_std': d_std      # kept (optional)
            })

df = pd.DataFrame(records)
print("✅ Simulation complete:", df.shape, "rows")

# -----------------------------
# 5. Plot style (BIG + readable)
# -----------------------------
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 15,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
    "legend.title_fontsize": 14,
})

colors = plt.cm.viridis(np.linspace(0, 1, len(mean_shifts)))
distribs = list(distributions.keys())
ensure_dir(EFFECT_SIZE_FIGURES_DIR)

# BIGGER panel geometry
SINGLE_PANEL_W = 8.5
SINGLE_PANEL_H = 6.0
LEGEND_W = 3.2

LINEWIDTH = 2.8
MARKERSIZE = 9
BAND_ALPHA = 0.22

fig, axes = plt.subplots(
    1, len(distribs),
    figsize=(SINGLE_PANEL_W * len(distribs) + LEGEND_W, SINGLE_PANEL_H),
    sharex=True, sharey=True
)
axes = np.atleast_1d(axes).flatten()

# -----------------------------
# 6. Robust symmetric y-limits
# -----------------------------
all_vals = df[['d_q25', 'd_q75']].to_numpy().ravel()
all_vals = all_vals[np.isfinite(all_vals)]

if all_vals.size > 0:
    max_abs = np.quantile(np.abs(all_vals), 0.99)
    ylim = max(0.5, max_abs * 1.15)
else:
    ylim = 1.0

# -----------------------------
# 7. Plot: median + IQR
# -----------------------------
legend_handles, legend_labels = None, None

for i, dist_name in enumerate(distribs):
    ax = axes[i]
    sub = df[df['dist'] == dist_name]

    for color, delta in zip(colors, mean_shifts):
        subset = sub[sub['delta'] == delta].sort_values('n')
        if subset.empty:
            continue

        n_vals = subset['n'].to_numpy()
        d_med  = subset['d_med'].to_numpy()
        d_q25  = subset['d_q25'].to_numpy()
        d_q75  = subset['d_q75'].to_numpy()

        ax.plot(
            n_vals, d_med,
            marker='o',
            markersize=MARKERSIZE,
            linewidth=LINEWIDTH,
            color=color,
            label=f'Δ={delta}'
        )

        ax.fill_between(
            n_vals, d_q25, d_q75,
            alpha=BAND_ALPHA,
            color=color
        )

    # Reference lines
    ax.axhline(0.0, color='black', lw=1.4, alpha=0.8)
    ax.axhline(0.2, color='gray', lw=1.0, ls='--', alpha=0.5)
    ax.axhline(-0.2, color='gray', lw=1.0, ls='--', alpha=0.5)

    ax.set_xscale('log')
    ax.set_ylim(-ylim, ylim)
    ax.set_title(dist_name.capitalize())
    ax.set_xlabel('Sample size (log scale)')
    ax.grid(True, which="both", ls="--", lw=0.6, alpha=0.7)

    if i == 0:
        ax.set_ylabel("Cohen's d (median ± IQR)")

    if legend_handles is None:
        legend_handles, legend_labels = ax.get_legend_handles_labels()

# Global legend
fig.legend(
    legend_handles, legend_labels,
    title='Parameter shift Δ',
    loc='center right',
    bbox_to_anchor=(1.03, 0.5),
    frameon=True
)

fig.suptitle(
    "Stability of Cohen’s d Across Sample Sizes (Median ± IQR)",
    fontsize=18
)

fig.tight_layout(rect=[0, 0, 0.95, 0.95])

plt.savefig(EFFECT_SIZE_FIGURES_DIR / "cohens_d_stability_BIG.png", dpi=300)
plt.savefig(EFFECT_SIZE_FIGURES_DIR / "cohens_d_stability_BIG.pdf")
plt.show()


In [ ]:
from experiments.paths import EXPERIMENTS_DIR

df.to_csv(EXPERIMENTS_DIR / "cohen_sensitiv.csv", index=False)
